<a href="https://colab.research.google.com/github/NidhiCN24/Festiva_Projects_10000053456345432365634/blob/main/Festiva_Planner_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Jai Ganesha
#Installing dependencies
!pip -q install fastapi uvicorn pyngrok nest-asyncio gradio faiss-cpu sentence-transformers transformers langchain langchain-community scikit-learn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
#Importing libraries
import os
import json
import random
import asyncio
import threading
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import classification_report
import faiss
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import gradio as gr

In [3]:
#mounting drive
from google.colab import drive
drive.mount('/content/drive')

#printing the first 5 rows
df = pd.read_csv("EventManagmentDataSet.csv")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   SNO                        Event Faculty_id   Faculty_name  \
0    1         make a creative logo  FAC010102       John Doe   
1    2    Presentiong ideas on MSME  FAC010070     Jane Smith   
2    3  Webinar on Web-Technologies  FAC010103  Alice Johnson   
3    4             Group Discussion  FAC010077      Bob Brown   
4    5               Session on SIH  FAC010060    Carol Davis   

   Faculty_Designation  Registers             Type  Request for Booking  \
0  Associate Professor        100       Important                     1   
1  Associate Professor        105  Very important                     1   
2  Associate Professor         77           Normal                    1   
3  Associate Professor         60       Important                     1   
4  Associate Professor        111  Very important                     1   

   Request for Cancelling  Re

In [4]:
#Checking for missing values
df.isnull().sum()

,0
SNO,0
Event,0
Faculty_id,0
Faculty_name,0
Faculty_Designation,0
Registers,0
Type,0
Request for Booking,0
Request for Cancelling,0
Requested Room No,0


In [5]:
#creating synthetic dataset
cities = ["Bangalore","Mumbai","Delhi","Hyderabad"]
event_types = ["wedding","corporate","birthday"]

rows = []

for _ in range(1500):

    et = random.choice(event_types)
    city = random.choice(cities)
    budget = random.randint(10000, 5000000)

    splits = np.random.dirichlet([3,4,2,1,1])

    venue_p = splits[0]
    catering_p = splits[1]
    decor_p = splits[2]
    entertainment_p = splits[3]
    logistics_p = splits[4]

    rows.append([
        et,
        city,
        budget,
        venue_p,
        catering_p,
        decor_p,
        entertainment_p,
        logistics_p
    ])

budget_df = pd.DataFrame(
    rows,
    columns=[
        "event_type",
        "city",
        "budget",
        "venue_p",
        "catering_p",
        "decor_p",
        "entertainment_p",
        "logistics_p"
    ]
)

budget_df.head()

,event_type,city,budget,venue_p,catering_p,decor_p,entertainment_p,logistics_p
0,wedding,Mumbai,1623238,0.122696,0.585253,0.041109,0.090348,0.160594
1,wedding,Mumbai,3705626,0.441156,0.072825,0.336535,0.124079,0.025404
2,birthday,Mumbai,4794741,0.236295,0.667682,0.043244,0.025040,0.027740
3,corporate,Mumbai,327997,0.222272,0.317953,0.066883,0.112055,0.280836
4,birthday,Hyderabad,3360215,0.272718,0.318009,0.197037,0.032271,0.179966


In [6]:
#training budget optimizer
X = pd.get_dummies(
    budget_df[["event_type","city","budget"]]
)

Y = budget_df[
    [
        "venue_p",
        "catering_p",
        "decor_p",
        "entertainment_p",
        "logistics_p"
    ]
]

budget_model = MultiOutputRegressor(
    RandomForestRegressor()
)

budget_model.fit(X, Y)

print("Budget model trained")

Budget model trained


In [7]:
#Event type classifier
texts = df["Event"].astype(str)
labels = np.random.choice(event_types, len(df))

vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(texts)

clf = LogisticRegression()
clf.fit(X_text, labels)

print("NLP model ready")

NLP model ready


In [8]:
# Building event-tagged RAG knowledge base

knowledge_data = [

    {
        "event_type": "wedding",
        "text": "Wedding should start planning 6 months early."
    },

    {
        "event_type": "wedding",
        "text": "Venue booking should be finalized first."
    },

    {
        "event_type": "wedding",
        "text": "Catering should be booked at least one month before."
    },

    {
        "event_type": "corporate",
        "text": "Corporate events require agenda planning."
    },

    {
        "event_type": "corporate",
        "text": "Speaker schedules should be finalized early."
    },

    {
        "event_type": "corporate",
        "text": "AV equipment testing is mandatory."
    },

    {
        "event_type": "birthday",
        "text": "Birthday events need guest list and entertainment."
    },

    {
        "event_type": "birthday",
        "text": "Cake booking should be completed one week before."
    },

    {
        "event_type": "birthday",
        "text": "Theme decoration should be finalized early."
    }
]

knowledge_texts = [
    item["text"]
    for item in knowledge_data
]

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedder.encode(
    knowledge_texts
)

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(
    np.array(embeddings)
)

print("RAG ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG ready


In [9]:
#retrieval function
def retrieve_knowledge(event_type):

    # Step 1: filter only this event type
    filtered_data = [

        item

        for item in knowledge_data

        if item["event_type"] == event_type
    ]


    # Step 2: extract text
    filtered_texts = [

        item["text"]

        for item in filtered_data
    ]


    # Step 3: embed only filtered text
    filtered_embeddings = embedder.encode(
        filtered_texts
    )


    # Step 4: local FAISS search
    local_index = faiss.IndexFlatL2(
        filtered_embeddings.shape[1]
    )

    local_index.add(
        np.array(filtered_embeddings)
    )


    # Step 5: search inside this event only
    query_embedding = embedder.encode(
        [event_type]
    )

    D, I = local_index.search(
        np.array(query_embedding),
        min(2, len(filtered_texts))
    )


    return [

        filtered_texts[i]

        for i in I[0]
    ]

In [10]:
#Planner agent
def planner_agent(event_type):

    plans = {

        "wedding": {
            "timeline": [
                "6 months before: venue",
                "3 months before: vendors",
                "1 month before: confirmations",
                "Event day: execution"
            ],

            "checklist": [
                "venue",
                "decor",
                "catering"
            ],

            "vendors": [
                "venue",
                "photography",
                "catering"
            ]
        },

        "corporate": {
            "timeline": [
                "3 months before: define agenda",
                "2 months before: venue",
                "1 month before: speakers",
                "Event day: execution"
            ],

            "checklist": [
                "venue",
                "AV",
                "branding"
            ],

            "vendors": [
                "conference hall",
                "audio-video",
                "branding"
            ]
        },

        "birthday": {
            "timeline": [
                "1 month before: venue",
                "2 weeks before: cake",
                "1 week before: invitations",
                "Event day: celebration"
            ],

            "checklist": [
                "venue",
                "cake",
                "decor",
                "music"
            ],

            "vendors": [
                "cake shop",
                "decorator",
                "DJ",
                "photography"
            ]
        }
    }

    return plans[event_type]

In [11]:
#budget agent
def budget_agent(event_type, city, budget):

    inp = pd.DataFrame([
        {
            "event_type": event_type,
            "city": city,
            "budget": budget
        }
    ])

    inp = pd.get_dummies(inp)

    inp = inp.reindex(
        columns=X.columns,
        fill_value=0
    )

    proportions = budget_model.predict(inp)[0]

    proportions = np.maximum(
        proportions,
        0
    )

    proportions = proportions / proportions.sum()

    allocations = proportions * budget

    return {
        "venue": int(allocations[0]),
        "catering": int(allocations[1]),
        "decor": int(allocations[2]),
        "entertainment": int(allocations[3]),
        "logistics": int(allocations[4]),
        "total": int(allocations.sum())
    }

In [12]:
#knowledge agent
def knowledge_agent(event_type):

    return retrieve_knowledge(
        event_type
    )

In [13]:
#multi agent orchestrator
def orchestrator(event_type, city, budget):

    plan = planner_agent(event_type)
    budget_data = budget_agent(
        event_type,
        city,
        budget
    )

    knowledge = knowledge_agent(
        event_type
    )

    return {
        "plan":plan,
        "budget":budget_data,
        "knowledge":knowledge
    }

In [14]:
#FastAPI backend
app = FastAPI()

class RequestModel(BaseModel):
    event_type:str
    city:str
    budget:int

@app.post("/plan")
def create_plan(req: RequestModel):
    return orchestrator(
        req.event_type,
        req.city,
        req.budget
    )

In [15]:
#running fastAPI(backend)
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

threading.Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    ),
    daemon=True
).start()

print("FastAPI running inside Colab")

FastAPI running inside Colab


In [16]:
#gradio UI
def ui_runner(
    event_type,
    city,
    budget
):

    result = orchestrator(
        event_type,
        city,
        int(budget)
    )

    return json.dumps(
        result,
        indent=2
    )

ui = gr.Interface(
    fn=ui_runner,
    inputs=[
        gr.Dropdown(event_types),
        gr.Dropdown(cities),
        gr.Number()
    ],
    outputs = gr.Textbox(lines=30),
    title = "Festiva Planner AI"
    )

ui.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2a39f9efc48aeeaab7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2a39f9efc48aeeaab7.gradio.live
